# Speech-Toolformer: Voice Banking Assistant Demo

**DLS MIPT Final Project — Speech, Fall 2025**

This notebook demonstrates a zero-shot voice banking assistant built on **Qwen2.5-Omni-7B**.
The user says a phrase like *"Transfer five hundred to Mom"* — the system transcribes it and returns a structured tool call:

```json
{"tool_name": "transfer_money", "arguments": {"recipient": "Mom", "amount": 500.0}}
```

Four pipelines are evaluated:
- **A** — Text Oracle (GT text → JSON)
- **B** — ASR only (audio → transcript, WER)
- **C** — Direct (audio → JSON in one step)
- **D** — Cascaded (audio → ASR → JSON)


## 1. Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # adjust to your GPU

import json
import re
import torch
from collections import Counter
from IPython.display import Audio, display, Markdown

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

## 2. Load Model

In [ ]:
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"
print(f"Loading {MODEL_ID}...")

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Model loaded.")

## 3. Helper Functions

In [ ]:
SYSTEM_PROMPT = """You are a banking assistant.
You must NOT generate audio or speech output. Generate TEXT ONLY.

Available tools:
1. transfer_money(recipient: str, amount: float)
2. get_exchange_rate(currency_from: str, currency_to: str)

Output ONLY valid JSON format: {"tool_name": "...", "arguments": {...}}.
If the request is unrelated to these tools, return null."""

ASR_PROMPT = "Please transcribe the audio accurately."


def generate_omni(conversation: list) -> str:
    """Run the model, return only newly generated text."""
    text = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(
        text=text, audio=audios, images=images, videos=videos,
        return_tensors="pt", padding=True, use_audio_in_video=False,
    )
    inputs = inputs.to(model.device).to(model.dtype)
    input_len = inputs.input_ids.shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, use_audio_in_video=False, return_audio=False, max_new_tokens=128,
        )
        text_ids = out[0] if isinstance(out, tuple) else out
    new_tokens = text_ids[0][input_len:]
    return processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def extract_json(text: str):
    """Extract JSON dict / None / 'PARSE_ERROR' from model output."""
    try:
        m = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
        if m:
            return json.loads(m.group(1))
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            return json.loads(m.group(0))
        if "null" in text.lower():
            return None
        return "PARSE_ERROR"
    except Exception:
        return "PARSE_ERROR"


def is_hallucination(text: str) -> bool:
    """Detect looping / non-Latin hallucinations in ASR output."""
    if not isinstance(text, str) or not text.strip():
        return True
    if len(re.findall(r'[^\x00-\x7F]', text)) > 3:
        return True
    words = text.split()
    for i in range(len(words) - 3):
        if len(set(words[i:i+4])) == 1:
            return True
    if len(words) >= 9:
        trigrams = [tuple(words[i:i+3]) for i in range(len(words) - 2)]
        if Counter(trigrams).most_common(1)[0][1] > 2:
            return True
    if len(words) > 40:
        return True
    return False


def extract_clean_transcript(text: str) -> str:
    """Strip Qwen's verbose wrapper and return the bare transcribed phrase."""
    if not isinstance(text, str) or not text.strip():
        return ""
    if is_hallucination(text):
        return ""
    m = re.search(r"'([^']{3,})'", text)
    if m:
        return m.group(1).strip(" .")
    m = re.search(r'"([^"]{3,})"', text)
    if m:
        return m.group(1).strip(" .")
    m = re.search(r'(?:is|are)\s*:\s*(.{5,})$', text.strip(), re.IGNORECASE)
    if m:
        return m.group(1).strip("'.\" ")
    service = ("original content", "transcription", "audio is", "the content")
    if not any(sw in text.lower() for sw in service):
        return text.strip(" .")
    return ""


print("Helper functions defined.")

## 4. Pick a Sample from the Dataset

In [ ]:
DATASET_FILE = "data/dataset_audio_Cameron_Russell_115.json"

with open(DATASET_FILE, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Dataset: {len(dataset)} samples")

# Pick one positive sample (transfer_money) and one negative sample (null)
positive_sample = next(s for s in dataset if s.get("label") is not None)
negative_sample = next(s for s in dataset if s.get("label") is None)

# Use the positive sample for the demo
sample = positive_sample

print(f"\n--- Demo Sample ---")
print(f"Text (GT): {sample['text']}")
print(f"Label:     {json.dumps(sample['label'], ensure_ascii=False)}")
print(f"Audio:     {sample['audio_path']}")

In [ ]:
# Play the audio
display(Audio(sample["audio_path"]))

## 5. Pipeline A — Text Oracle

Ground-truth text → JSON. Upper bound on LLM comprehension quality.

In [ ]:
conv_a = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [{"type": "text", "text": sample["text"]}]},
]

raw_a = generate_omni(conv_a)
json_a = extract_json(raw_a)

print(f"Input text: {sample['text']}")
print(f"Raw output: {raw_a}")
print(f"Parsed:     {json_a}")
print(f"Ground truth: {sample['label']}")

## 6. Pipeline B — ASR (Audio → Transcript)

Audio → text transcription. Evaluated with WER.

In [ ]:
conv_b = [
    {"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant."}]},
    {"role": "user",   "content": [
        {"type": "audio", "audio": sample["audio_path"]},
        {"type": "text",  "text": ASR_PROMPT},
    ]},
]

raw_b = generate_omni(conv_b)
clean_b = extract_clean_transcript(raw_b)

print(f"GT text:         {sample['text']}")
print(f"Raw ASR output:  {raw_b}")
print(f"Clean transcript:{clean_b}")
print(f"Hallucination?   {is_hallucination(raw_b)}")

## 7. Pipeline C — Direct (Audio → JSON)

Audio directly to JSON tool call in a single model pass.

In [ ]:
conv_c = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [
        {"type": "audio", "audio": sample["audio_path"]},
        {"type": "text",  "text": "Extract the intent."},
    ]},
]

raw_c = generate_omni(conv_c)
json_c = extract_json(raw_c)

print(f"Raw output: {raw_c}")
print(f"Parsed:     {json_c}")
print(f"Ground truth: {sample['label']}")

## 8. Pipeline D — Cascaded (ASR → JSON)

Audio → clean transcript (Pipeline B) → JSON (second LLM call).

In [ ]:
# Pipeline D uses the clean transcript extracted in Pipeline B
d_input = clean_b if clean_b else "[ASR FAILED]"

conv_d = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [{"type": "text", "text": d_input}]},
]

raw_d = generate_omni(conv_d)
json_d = extract_json(raw_d)

print(f"D input (clean ASR): {d_input}")
print(f"Raw output:          {raw_d}")
print(f"Parsed:              {json_d}")
print(f"Ground truth:        {sample['label']}")

## 9. Summary for This Sample

In [ ]:
def compare_json(pred, truth) -> bool:
    if pred == "PARSE_ERROR": return False
    if pred is None and truth is None: return True
    if pred is None or truth is None: return False
    if not isinstance(pred, dict) or not isinstance(truth, dict): return False
    if pred.get("tool_name") != truth.get("tool_name"): return False
    pred_args = pred.get("arguments", {})
    truth_args = truth.get("arguments", {})
    for key, t_val in truth_args.items():
        p_val = pred_args.get(key)
        if isinstance(t_val, (int, float)) and isinstance(p_val, (int, float)):
            if abs(t_val - p_val) > 0.1: return False
        elif isinstance(t_val, str) and isinstance(p_val, str):
            if t_val.lower().strip() != p_val.lower().strip(): return False
        else:
            if str(t_val) != str(p_val): return False
    return True


gt = sample["label"]

print(f"GT text : {sample['text']}")
print(f"GT label: {gt}")
print()
print(f"Pipeline A → {json_a}   [{'✓' if compare_json(json_a, gt) else '✗'}]")
print(f"Pipeline B → '{clean_b}' (ASR transcript)")
print(f"Pipeline C → {json_c}   [{'✓' if compare_json(json_c, gt) else '✗'}]")
print(f"Pipeline D → {json_d}   [{'✓' if compare_json(json_d, gt) else '✗'}]")

## 10. Simulated Tool Execution

In production, the parsed JSON would trigger a real banking API call. Here we mock it.

In [ ]:
# Mock banking tools
def transfer_money(recipient: str, amount: float) -> dict:
    return {
        "status": "success",
        "transaction_id": "TXN-20250301-0042",
        "recipient": recipient,
        "amount": amount,
        "currency": "USD",
        "message": f"Transferred ${amount:.2f} to {recipient}.",
    }

def get_exchange_rate(currency_from: str, currency_to: str) -> dict:
    rates = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "GBP"): 0.79}
    rate = rates.get((currency_from.upper(), currency_to.upper()), None)
    return {
        "status": "success" if rate else "error",
        "from": currency_from,
        "to": currency_to,
        "rate": rate,
    }

TOOLS = {
    "transfer_money": transfer_money,
    "get_exchange_rate": get_exchange_rate,
}

def execute_tool_call(tool_call) -> str:
    if tool_call is None:
        return "No action taken (request not related to available tools)."
    if tool_call == "PARSE_ERROR":
        return "ERROR: model output could not be parsed as JSON."
    fn = TOOLS.get(tool_call.get("tool_name"))
    if fn is None:
        return f"ERROR: unknown tool '{tool_call.get('tool_name')}'"
    result = fn(**tool_call["arguments"])
    return json.dumps(result, indent=2, ensure_ascii=False)


# Execute Pipeline C result
print("=== Executing Pipeline C output ===")
print(f"Tool call: {json_c}")
print()
print(execute_tool_call(json_c))

In [ ]:
# Also demonstrate a negative case
print("=== Negative sample demo ===")
print(f"Text: {negative_sample['text']}")
print(f"GT:   {negative_sample['label']}")
print()

conv_neg = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [{"type": "text", "text": negative_sample["text"]}]},
]
raw_neg = generate_omni(conv_neg)
json_neg = extract_json(raw_neg)
print(f"Raw:    {raw_neg}")
print(f"Parsed: {json_neg}")
print()
print(execute_tool_call(json_neg))

## 11. Full Dataset Metrics (pre-computed)

Metrics computed from `results/results_omni_7b_v3.json` (511 samples).

In [ ]:
from jiwer import wer as compute_wer

RESULTS_FILE = "results/results_omni_7b_v3.json"

with open(RESULTS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples from {RESULTS_FILE}")


def compute_pipeline_metrics(data: list, pipeline_key: str) -> dict:
    tp = fp = fn = tn = parse_errors = 0
    for item in data:
        truth = item.get("ground_truth")
        pred  = item.get(pipeline_key)
        if pred == "PARSE_ERROR":
            parse_errors += 1
        is_pos_truth = truth is not None
        is_pos_pred  = pred is not None and pred != "PARSE_ERROR"
        correct = compare_json(pred, truth)
        if is_pos_truth:
            tp += 1 if correct else 0
            fn += 0 if correct else 1
        else:
            fp += 1 if is_pos_pred else 0
            tn += 0 if is_pos_pred else 1
    total = len(data)
    return {
        "accuracy":      (tp + tn) / total,
        "precision":     tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        "recall":        tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        "far":           fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        "parsable_rate": (total - parse_errors) / total,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


def normalize_for_wer(text: str) -> str:
    return " ".join(re.sub(r"[^\w\s]", "", text).lower().split())


m_a = compute_pipeline_metrics(data, "pipeline_a_json")
m_c = compute_pipeline_metrics(data, "pipeline_c_json")
m_d = compute_pipeline_metrics(data, "pipeline_d_json")

# WER for Pipeline B
wer_refs, wer_hyps, hall_count = [], [], 0
for item in data:
    ref = normalize_for_wer(item.get("input_text_gt", ""))
    if not ref.strip():
        continue
    if is_hallucination(item.get("pipeline_b_transcript", "")):
        hall_count += 1
        continue
    hyp_raw = item.get("pipeline_b_transcript_clean", "")
    if is_hallucination(hyp_raw):
        hall_count += 1
        continue
    wer_refs.append(ref)
    wer_hyps.append(normalize_for_wer(hyp_raw))

final_wer = compute_wer(wer_refs, wer_hyps) if wer_refs else float("nan")

print()
print(f"{'Pipeline':<30} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'FAR':>7}")
print("-" * 68)
for name, m in [("A  Text Oracle", m_a), ("C  Direct (audio→JSON)", m_c), ("D  Cascaded (ASR→JSON)", m_d)]:
    print(f"  {name:<28} {m['accuracy']:>8.2%}  {m['precision']:>9.3f}  {m['recall']:>7.3f}  {m['far']:>6.3f}")
print("-" * 68)
print(f"  {'B  ASR only (WER)':<28} {final_wer:>8.2%}  (word error rate, {len(wer_refs)} samples)")
print()
print("Note: WER is elevated due to digit↔word mismatch ('1500' vs 'fifteen hundred').")
print("Semantic WER is estimated at ~20-30%.")

## 12. Error Breakdown (Pipeline D — recommended for banking)

In [ ]:
def categorize_error(pred, truth) -> str:
    if compare_json(pred, truth):   return "correct"
    if pred == "PARSE_ERROR":       return "parse_error"
    is_pos_truth = truth is not None
    is_pos_pred  = pred is not None and pred != "PARSE_ERROR"
    if not is_pos_truth and is_pos_pred:  return "false_positive"
    if is_pos_truth and not is_pos_pred:  return "false_negative"
    if not isinstance(pred, dict) or not isinstance(truth, dict): return "wrong_format"
    pa = pred.get("arguments", {})
    ta = truth.get("arguments", {})
    wr = isinstance(ta.get("recipient"), str) and isinstance(pa.get("recipient"), str) and \
         ta["recipient"].lower().strip() != pa["recipient"].lower().strip()
    wa = True
    if isinstance(ta.get("amount"), (int, float)) and isinstance(pa.get("amount"), (int, float)):
        wa = abs(ta["amount"] - pa["amount"]) > 0.1
    elif ta.get("amount") == pa.get("amount"):
        wa = False
    if wr and wa:  return "both_wrong"
    if wr:         return "wrong_recipient"
    if wa:         return "wrong_amount"
    if pred.get("tool_name") != truth.get("tool_name"): return "wrong_tool"
    return "other_mismatch"


counts = Counter()
for item in data:
    counts[categorize_error(item.get("pipeline_d_json"), item.get("ground_truth"))] += 1

total = len(data)
print("Pipeline D — error breakdown (511 samples):")
print()
for cat, cnt in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "█" * (cnt * 40 // total)
    print(f"  {cat:<20} {cnt:>4}  {cnt/total:>6.1%}  {bar}")

## 13. Conclusions

| Pipeline | Accuracy | Precision | Recall | FAR | Recommendation |
|----------|:--------:|:---------:|:------:|:---:|----------------|
| **A** — Text Oracle | **98.83%** | 0.997 | 0.988 | 0.009 | Upper bound |
| **B** — ASR | WER **66.74%** | — | — | — | Metric only |
| **C** — Direct | 68.30% | 0.992 | 0.601 | 0.018 | Simpler pipeline |
| **D** — Cascaded | 67.51% | **1.000** | 0.586 | **0.000** | **Banking use** |

**Key findings:**

1. **Zero-shot works** — Qwen2.5-Omni-7B achieves ~98.8% accuracy on text input with no fine-tuning.
2. **Modality gap is ~30pp** — moving from text to audio drops accuracy from 98.8% to ~68%. Main causes: ASR errors (digit↔word mismatch, first-word clipping from XTTS-v2).
3. **Pipeline D is preferred for banking** — precision=1.000 and FAR=0.000 mean zero false transactions on 511 samples. Pipeline C has slightly better recall but made 2 false transfers.
4. **C ≈ D in accuracy** (gap 0.78%) — the cascaded approach is just as accurate but safer.

**Remaining gap from text oracle to audio (~30pp)** is primarily:
- Digit format mismatch ("1500" in text vs. "fifteen hundred" in speech)
- TTS first-word onset clipping ("Send" → "then/sent", 36/511 cases)
- Genuine ASR errors on ambiguous names/amounts
